# CNN CUDA Case Study: Full MNIST

A small MNIST classifier, trained for 50 epochs per implementation. This notebook
uses GitHub as the source of truth and compares a CPU CNN with custom CUDA,
direct cuDNN, and PyTorch on one GPU and two GPUs using DataParallel.

Select **GPU T4 x2** and enable Internet. Every comparison uses the same global
batch of 64, the same initial weights, seed and split. Each GPU receives 32 images
in the two-GPU training runs. A batch of one uses only one replica.


In [ ]:
# First cell: clean reset and exact GitHub synchronization.
from pathlib import Path
import json, os, shutil, subprocess, sys

REPOSITORY = "https://github.com/saltypal/CNN-CUDA-Case-Study.git"
SOURCE_REF = "main"  # Replace with an exact commit to reproduce an earlier run.
ENVIRONMENT = "kaggle" if Path("/kaggle").exists() else "colab" if Path("/content").exists() else "local"
PROJECT = Path("/kaggle/working/cnn-cuda-case-study-source") if ENVIRONMENT == "kaggle" else Path("/tmp/cnn-cuda-case-study-source")
RESULTS = Path("/kaggle/working/cnn-cuda-case-study-full-mnist-results") if ENVIRONMENT == "kaggle" else Path(os.environ.get("CNN_RESULTS_DIR", "/tmp/cnn-cuda-case-study-full-mnist-results"))
PROJECT, RESULTS = PROJECT.resolve(), RESULTS.resolve()
assert PROJECT.name == "cnn-cuda-case-study-source"
assert PROJECT.parent in [Path("/kaggle/working"), Path("/tmp")]
assert RESULTS != PROJECT and PROJECT not in RESULTS.parents
if ENVIRONMENT == "colab" and str(RESULTS).startswith("/tmp"):
    raise ValueError("Set CNN_RESULTS_DIR to mounted Drive or another persistent destination before running.")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", SOURCE_REF], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "source_commit.txt").write_text(COMMIT + "\n")
print("Environment:", ENVIRONMENT, "Git commit:", COMMIT, "Results:", RESULTS)


## Hardware and dependencies

The custom kernels require a Linux C++ compiler and nvcc. The direct cuDNN adapter
uses the cuDNN library supplied with the runtime's PyTorch installation. Keep the
runtime's working GPU stack. CPU and custom CUDA kernels use FP32. No CUDA kernel
runs on a TPU. A TPU baseline requires separate PyTorch/XLA handling.


In [ ]:
import importlib.util
for package in ("torch", "numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda, "cuDNN:", torch.backends.cudnn.version())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index), torch.cuda.get_device_capability(index))
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("This requested comparison needs two GPUs. Select Kaggle GPU T4 x2.")
if shutil.which("nvcc") is None or shutil.which("g++") is None:
    raise RuntimeError("The GPU notebook needs nvcc and a Linux C++ compiler.")
print(subprocess.check_output(["nvcc", "--version"], text=True))


## Controlled experiment

The network has one four-filter 3x3 valid convolution, ReLU, 2x2 max pooling,
676 flattened features and ten logits. It has 6,810 parameters. We use
all 60,000 images from MNIST's training collection, using 48,000 for training
and 12,000 for validation, plus all 10,000 official test images. Proportional
stratification retains every image and preserves the digit distribution.

Seven training runs cover CPU PyTorch, custom CUDA on one/two GPUs, direct cuDNN
on one/two GPUs, and PyTorch CUDA on one/two GPUs. Native serial/OpenMP CPU and
shared CUDA convolution provide additional validation and operation benchmarks.


In [ ]:
config = json.loads((PROJECT / "config_full_mnist.json").read_text())
config["epochs"] = 50
config["batch_size"] = 64
config_path = RESULTS / "run_config_full_mnist.json"
config_path.write_text(json.dumps(config, indent=2))
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT / "src")
environment["OMP_NUM_THREADS"] = str(config["cpu_threads"])
environment["MKL_NUM_THREADS"] = str(config["cpu_threads"])
environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environment["PYTHONUNBUFFERED"] = "1"
print(json.dumps(config, indent=2))


## Validate, train, benchmark and save

Validation runs before training and checks forward results and all convolution
gradients, including two-GPU scatter/gather and gradient reduction. Every run
saves all epoch records, exact splits, environment, source hash and checkpoints.

Training timing excludes validation, checkpoint I/O, warmup and initial transfer.
Those times remain in the detailed outputs. Single-convolution and CNN inference
timing use warmed repeated samples and completed GPU work.


In [ ]:
command = [sys.executable, "-m", "hpc_cnn.run", "--config", str(config_path), "--run-root", str(RESULTS), "--skip-completed"]
print("Run:", " ".join(command))
subprocess.run(command, cwd=PROJECT, env=environment, check=True)

subprocess.run([sys.executable, str(PROJECT / "scripts" / "audit_results.py"), "--run-root", str(RESULTS), "--source-ref", COMMIT], cwd=PROJECT, env=environment, check=True)


In [ ]:
from IPython.display import display, Image, Markdown
display(Markdown((RESULTS / "REPORT.md").read_text()))
display(Image(filename=str(RESULTS / "performance.png")))
display(Image(filename=str(RESULTS / "learning_curves.png")))


## Demo and persistent artifacts

The demo uses the final checkpoints and the same held-out images for each engine.
Its cold startup time is separate from warmed benchmark latency. Saving a Kaggle
notebook version with outputs retains `/kaggle/working` artifacts. In an interactive
session, use Save Version with outputs before ending the runtime.


In [ ]:
subprocess.run([sys.executable, "-m", "hpc_cnn.demo", "--config", str(config_path), "--run-root", str(RESULTS)], cwd=PROJECT, env=environment, check=True)
archive = shutil.make_archive("/kaggle/working/cnn-cuda-case-study-full-mnist-results", "zip", root_dir=RESULTS) if ENVIRONMENT == "kaggle" else shutil.make_archive(str(RESULTS.parent / "cnn-cuda-case-study-full-mnist-results"), "zip", root_dir=RESULTS)
print("Saved result archive:", archive)
